In [47]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import copy
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from inverse_pinn_model import InversePINN

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

dtype = torch.float64

print("device =", device)

if torch.cuda.is_available():
    print(
        "GPU =",
        torch.cuda.get_device_name(0)
    )
    
dtype = torch.float64
SEED = 11

np.random.seed(SEED)
torch.manual_seed(SEED)


N_f = 2000
N_terminal = 500
N_boundary = 500

epsilon = 1e-4


hidden_width = 64
hidden_layers = 4

r_initial = 0.10
sigma_initial = 0.40


learning_rate = 1e-3
num_epochs = 30000

print_every = 1000
history_every = 100


lambda_data = 1.0
lambda_pde = 1.0
lambda_terminal = 0.01
lambda_boundary = 0.01


print("device =", device)

print()
print("N_f =", N_f)
print("N_terminal =", N_terminal)
print("N_boundary =", N_boundary)

print()
print("lambda_data =", lambda_data)
print("lambda_pde =", lambda_pde)
print("lambda_terminal =", lambda_terminal)
print("lambda_boundary =", lambda_boundary)

device = cpu
device = cpu

N_f = 2000
N_terminal = 500
N_boundary = 500

lambda_data = 1.0
lambda_pde = 1.0
lambda_terminal = 0.1
lambda_boundary = 0.1


In [49]:

# ------------------------------------------------------------
# Automatically locate NDX_call_data.csv
# ------------------------------------------------------------

from pathlib import Path

_filename = "NDX_call_data.csv"

_search_roots = [
    Path.cwd().resolve()
]

try:
    _notebook_dir = Path(__file__).resolve().parent
    _search_roots.append(_notebook_dir)
except NameError:
    pass


_candidates = []

for _root in _search_roots:

    _candidates.extend([
        _root / _filename,
        _root / "data" / _filename,
        _root.parent / _filename,
        _root.parent / "data" / _filename,
    ])


data_path = None


# First check common locations
for _candidate in _candidates:

    if _candidate.is_file():

        data_path = _candidate.resolve()
        break


# Then recursively search the current project
if data_path is None:

    for _root in _search_roots:

        try:

            _matches = list(
                _root.rglob(
                    _filename
                )
            )

        except Exception:

            _matches = []


        if _matches:

            data_path = (
                _matches[0]
                .resolve()
            )

            break


if data_path is None:

    raise FileNotFoundError(
        "\nCould not find NDX_call_data.csv.\n"
        "Make sure the CSV is included somewhere inside the downloaded project folder."
    )


print(
    "Using data file:",
    data_path
)


raw_df = pd.read_csv(
    data_path
)


# Remove automatically generated index columns
unnamed_columns = [
    col
    for col in raw_df.columns
    if str(col).startswith("Unnamed:")
]

if unnamed_columns:
    raw_df = raw_df.drop(
        columns=unnamed_columns
    )


raw_df = raw_df.reset_index(
    drop=True
)


# Save original row identity
raw_df["original_row"] = np.arange(
    len(raw_df)
)

if "market_price" not in raw_df.columns:

    if not {
        "best_bid",
        "best_offer"
    }.issubset(raw_df.columns):

        raise KeyError(
            "Need market_price or "
            "best_bid and best_offer."
        )

    raw_df["market_price"] = (
        raw_df["best_bid"]
        + raw_df["best_offer"]
    ) / 2.0


K = float(
    raw_df[
        "strike_price"
    ]
    .dropna()
    .iloc[0]
)

real_df = raw_df[
    [
        "original_row",
        "current_time",
        "spot_price",
        "market_price"
    ]
].copy()


# Remove invalid observations
real_df = (
    real_df
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .dropna()
)


real_df = real_df[
    (
        real_df["current_time"] >= 0.0
    )
    &
    (
        real_df["spot_price"] > 0.0
    )
    &
    (
        real_df["market_price"] >= 0.0
    )
].copy()


real_df = real_df.reset_index(
    drop=True
)


print(
    "Number of usable observations =",
    len(real_df)
)

print(
    "K =",
    K
)

Using data file: NDX_call_data.csv
Number of usable observations = 239
K = 15700.0


In [50]:

T = float(
    real_df["current_time"].max()
)


# PINN time variable
# t = T - time_to_maturity
real_df["t"] = (
    T
    - real_df["current_time"]
)


# Stock price
real_df["S"] = (
    real_df["spot_price"]
)


# Market option price
real_df["V"] = (
    real_df["market_price"]
)


t_min = 0.0
t_max = T

S_min = 0.0


S_data_max = float(
    real_df["S"].max()
)


# Enlarged PINN domain
S_max = (
    10.0
    * S_data_max
)


print("T =", T)

print(
    "Observed t range =",
    real_df["t"].min(),
    "to",
    real_df["t"].max()
)

print(
    "Observed S range =",
    real_df["S"].min(),
    "to",
    real_df["S"].max()
)

print(
    "PINN S_max =",
    S_max
)

T = 0.989041095890411
Observed t range = 0.0 to 0.9863013698630138
Observed S range = 11127.5703125 to 16573.33984375
PINN S_max = 165733.3984375


In [51]:
N_train = 200

rng = np.random.default_rng(SEED)

all_indices = np.arange(
    len(real_df)
)

shuffled_indices = rng.permutation(
    all_indices
)

train_indices = shuffled_indices[
    :N_train
]

validation_indices = shuffled_indices[
    N_train:
]


train_df = real_df.iloc[
    train_indices
].copy().reset_index(drop=True)


validation_df = real_df.iloc[
    validation_indices
].copy().reset_index(drop=True)


t_train = torch.tensor(
    train_df["t"].to_numpy(),
    dtype=dtype,
    device=device
).reshape(-1, 1)


S_train = torch.tensor(
    train_df["S"].to_numpy(),
    dtype=dtype,
    device=device
).reshape(-1, 1)


V_train = torch.tensor(
    train_df["V"].to_numpy(),
    dtype=dtype,
    device=device
).reshape(-1, 1)


u_train = (
    V_train / K
)


t_validation = torch.tensor(
    validation_df["t"].to_numpy(),
    dtype=dtype,
    device=device
).reshape(-1, 1)


S_validation = torch.tensor(
    validation_df["S"].to_numpy(),
    dtype=dtype,
    device=device
).reshape(-1, 1)


V_validation = torch.tensor(
    validation_df["V"].to_numpy(),
    dtype=dtype,
    device=device
).reshape(-1, 1)


u_validation = (
    V_validation / K
)


N_data = len(train_df)
N_validation = len(validation_df)


print(
    "Total observations =",
    len(real_df)
)

print(
    "Training observations =",
    N_data
)

print(
    "Validation observations =",
    N_validation
)

Total observations = 239
Training observations = 200
Validation observations = 39


In [52]:
def black_scholes_residual(
    model,
    t,
    S
):

    t_input = (
        t.clone()
        .detach()
        .requires_grad_(True)
    )


    S_input = (
        S.clone()
        .detach()
        .requires_grad_(True)
    )


    u = model(
        t_input,
        S_input
    )


    u_t = torch.autograd.grad(

        outputs=u,
        inputs=t_input,

        grad_outputs=torch.ones_like(
            u
        ),

        create_graph=True,
        retain_graph=True

    )[0]


    u_S = torch.autograd.grad(

        outputs=u,
        inputs=S_input,

        grad_outputs=torch.ones_like(
            u
        ),

        create_graph=True,
        retain_graph=True

    )[0]


    u_SS = torch.autograd.grad(

        outputs=u_S,
        inputs=S_input,

        grad_outputs=torch.ones_like(
            u_S
        ),

        create_graph=True,
        retain_graph=True

    )[0]


    residual = (

        u_t

        + 0.5
        * model.sigma ** 2
        * S_input ** 2
        * u_SS

        + model.r
        * S_input
        * u_S

        - model.r
        * u
    )


    return residual

In [53]:
# PINN loss
def compute_losses(

    model,

    t_train,
    S_train,
    u_train,

    t_f,
    S_f,

    t_terminal,
    S_terminal,

    t_boundary,
    S_boundary_left,
    S_boundary_right
):
    u_data_prediction = model(
        t_train,
        S_train
    )


    loss_data = torch.mean(

        (
            u_data_prediction
            - u_train
        ) ** 2
    )


    pde_prediction = (
        black_scholes_residual(
            model,
            t_f,
            S_f
        )
    )


    loss_pde = torch.mean(
        pde_prediction ** 2
    )

    u_terminal_prediction = model(
        t_terminal,
        S_terminal
    )


    u_terminal_true = (

        torch.relu(
            S_terminal
            - K
        )

        / K
    )


    loss_terminal = torch.mean(

        (
            u_terminal_prediction
            - u_terminal_true
        ) ** 2
    )


    u_left_prediction = model(
        t_boundary,
        S_boundary_left
    )


    u_left_true = torch.zeros_like(
        u_left_prediction
    )


    loss_boundary_left = torch.mean(

        (
            u_left_prediction
            - u_left_true
        ) ** 2
    )


    u_right_prediction = model(
        t_boundary,
        S_boundary_right
    )


    u_right_true = (

        S_boundary_right

        - K
        * torch.exp(

            -model.r

            * (
                T
                - t_boundary
            )
        )

    ) / K


    loss_boundary_right = torch.mean(

        (
            u_right_prediction
            - u_right_true
        ) ** 2
    )


    loss_boundary = (

        loss_boundary_left
        + loss_boundary_right
    )


    total_loss = (

        lambda_data
        * loss_data

        + lambda_pde
        * loss_pde

        + lambda_terminal
        * loss_terminal

        + lambda_boundary
        * loss_boundary
    )


    return {

        "total":
            total_loss,

        "data":
            loss_data,

        "pde":
            loss_pde,

        "terminal":
            loss_terminal,

        "boundary":
            loss_boundary,

        "boundary_left":
            loss_boundary_left,

        "boundary_right":
            loss_boundary_right
    }

In [54]:

# Reset seed so these points are reproducible
torch.manual_seed(SEED)


t_f = (

    t_min
    + epsilon

    + (
        t_max
        - t_min
        - 2.0 * epsilon
    )

    * torch.rand(
        N_f,
        1,
        dtype=dtype,
        device=device
    )
)


S_f = (

    S_min
    + epsilon

    + (
        S_max
        - S_min
        - 2.0 * epsilon
    )

    * torch.rand(
        N_f,
        1,
        dtype=dtype,
        device=device
    )
)

t_terminal = torch.full(

    (
        N_terminal,
        1
    ),

    fill_value=T,

    dtype=dtype,
    device=device
)


S_terminal = (

    S_min

    + (
        S_max
        - S_min
    )

    * torch.rand(
        N_terminal,
        1,
        dtype=dtype,
        device=device
    )
)


t_boundary = (

    t_min

    + (
        t_max
        - t_min
    )

    * torch.rand(
        N_boundary,
        1,
        dtype=dtype,
        device=device
    )
)


# Left boundary: S = 0

S_boundary_left = torch.full(

    (
        N_boundary,
        1
    ),

    fill_value=S_min,

    dtype=dtype,
    device=device
)


# Right boundary: S = S_max

S_boundary_right = torch.full(

    (
        N_boundary,
        1
    ),

    fill_value=S_max,

    dtype=dtype,
    device=device
)


print(
    "PDE t range:",
    float(t_f.min()),
    "to",
    float(t_f.max())
)

print(
    "PDE S range:",
    float(S_f.min()),
    "to",
    float(S_f.max())
)

print()

print(
    "Terminal S range:",
    float(S_terminal.min()),
    "to",
    float(S_terminal.max())
)

print()

print(
    "Right boundary S =",
    float(S_boundary_right[0])
)

PDE t range: 0.00037028630981712513 to 0.9883740397694238
PDE S range: 35.20510665675789 to 165715.4108931402

Terminal S range: 125.2145481055958 to 165091.5593422809

Right boundary S = 165733.3984375


In [55]:
torch.manual_seed(SEED)


base_model = InversePINN(

    t_min=t_min,
    t_max=t_max,

    S_min=S_min,
    S_max=S_max,

    hidden_width=hidden_width,
    hidden_layers=hidden_layers,

    r_initial=r_initial,
    sigma_initial=sigma_initial

).to(
    device=device,
    dtype=dtype
)


base_state_dict = copy.deepcopy(
    base_model.state_dict()
)


print(
    "Initial r =",
    base_model.r.item()
)

print(
    "Initial sigma =",
    base_model.sigma.item()
)


del base_model

Initial r = 0.10000000000000002
Initial sigma = 0.4


In [56]:
start_time = time.time()

model = InversePINN(

    t_min=t_min,
    t_max=t_max,

    S_min=S_min,
    S_max=S_max,

    hidden_width=hidden_width,
    hidden_layers=hidden_layers,

    r_initial=r_initial,
    sigma_initial=sigma_initial

).to(
    device=device,
    dtype=dtype
)


model.load_state_dict(
    copy.deepcopy(
        base_state_dict
    )
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate
)


history = {

    "epoch": [],

    "total_loss": [],

    "data_loss": [],
    "pde_loss": [],
    "terminal_loss": [],

    "boundary_loss": [],
    "boundary_left_loss": [],
    "boundary_right_loss": [],

    # Weighted components
    "weighted_data_loss": [],
    "weighted_pde_loss": [],
    "weighted_terminal_loss": [],
    "weighted_boundary_loss": [],

    "r_estimate": [],
    "sigma_estimate": []
}


print()
print("=" * 80)

print(
    "Training real-data inverse PINN"
)

print(
    f"Number of real observations = {N_data}"
)

print()

print(
    "Lambdas:"
)

print(
    f"data={lambda_data}, "
    f"pde={lambda_pde}, "
    f"terminal={lambda_terminal}, "
    f"boundary={lambda_boundary}"
)

print("=" * 80)


for epoch in range(
    1,
    num_epochs + 1
):

    model.train()


    optimizer.zero_grad(
        set_to_none=True
    )


    losses = compute_losses(

        model=model,

        t_train=t_train,
        S_train=S_train,
        u_train=u_train,

        t_f=t_f,
        S_f=S_f,

        t_terminal=t_terminal,
        S_terminal=S_terminal,

        t_boundary=t_boundary,
        S_boundary_left=S_boundary_left,
        S_boundary_right=S_boundary_right
    )


    losses[
        "total"
    ].backward()


    optimizer.step()

    if (

        epoch == 1

        or epoch
        % history_every
        == 0

        or epoch
        == num_epochs
    ):

        history[
            "epoch"
        ].append(
            epoch
        )


        history[
            "total_loss"
        ].append(
            losses[
                "total"
            ].item()
        )


        history[
            "data_loss"
        ].append(
            losses[
                "data"
            ].item()
        )


        history[
            "pde_loss"
        ].append(
            losses[
                "pde"
            ].item()
        )


        history[
            "terminal_loss"
        ].append(
            losses[
                "terminal"
            ].item()
        )


        history[
            "boundary_loss"
        ].append(
            losses[
                "boundary"
            ].item()
        )


        history[
            "boundary_left_loss"
        ].append(
            losses[
                "boundary_left"
            ].item()
        )


        history[
            "boundary_right_loss"
        ].append(
            losses[
                "boundary_right"
            ].item()
        )


        history[
            "weighted_data_loss"
        ].append(
            lambda_data
            * losses[
                "data"
            ].item()
        )


        history[
            "weighted_pde_loss"
        ].append(
            lambda_pde
            * losses[
                "pde"
            ].item()
        )


        history[
            "weighted_terminal_loss"
        ].append(
            lambda_terminal
            * losses[
                "terminal"
            ].item()
        )


        history[
            "weighted_boundary_loss"
        ].append(
            lambda_boundary
            * losses[
                "boundary"
            ].item()
        )


        history[
            "r_estimate"
        ].append(
            model.r.item()
        )


        history[
            "sigma_estimate"
        ].append(
            model.sigma.item()
        )


    if (

        epoch == 1

        or epoch
        % print_every
        == 0

        or epoch
        == num_epochs
    ):

        print(

            f"Epoch {epoch:5d} | "

            f"Total: "
            f"{losses['total'].item():.6e} | "

            f"Data: "
            f"{losses['data'].item():.3e} | "

            f"PDE: "
            f"{losses['pde'].item():.3e} | "

            f"Terminal: "
            f"{losses['terminal'].item():.3e} | "

            f"Boundary: "
            f"{losses['boundary'].item():.3e} | "

            f"r: "
            f"{model.r.item():.8f} | "

            f"sigma: "
            f"{model.sigma.item():.8f}"
        )


runtime_minutes = (
    time.time()
    - start_time
) / 60.0


r_pinn = model.r.item()
sigma_pinn = model.sigma.item()


print()
print("=" * 80)

print("TRAINING COMPLETE")

print()
print(
    f"r_PINN = {r_pinn:.10f}"
)

print(
    f"sigma_PINN = {sigma_pinn:.10f}"
)

print()

print(
    f"Runtime = {runtime_minutes:.2f} minutes"
)

print("=" * 80)


Training real-data inverse PINN
Number of real observations = 200

Lambdas:
data=1.0, pde=1.0, terminal=0.1, boundary=0.1
Epoch     1 | Total: 1.139675e+01 | Data: 1.104e-01 | PDE: 1.512e-03 | Terminal: 2.696e+01 | Boundary: 8.589e+01 | r: 0.09990488 | sigma: 0.39967043
Epoch  1000 | Total: 1.620854e-02 | Data: 2.013e-03 | PDE: 9.744e-04 | Terminal: 4.969e-02 | Boundary: 8.253e-02 | r: 0.08369804 | sigma: 0.32583920
Epoch  2000 | Total: 8.028114e-04 | Data: 1.642e-04 | PDE: 2.857e-04 | Terminal: 2.962e-03 | Boundary: 5.680e-04 | r: 0.08187938 | sigma: 0.31798163
Epoch  3000 | Total: 4.325503e-03 | Data: 2.773e-03 | PDE: 5.512e-04 | Terminal: 1.783e-03 | Boundary: 8.228e-03 | r: 0.08072348 | sigma: 0.31298715
Epoch  4000 | Total: 3.226979e-04 | Data: 1.351e-04 | PDE: 1.130e-04 | Terminal: 7.125e-04 | Boundary: 3.305e-05 | r: 0.07911762 | sigma: 0.30596926
Epoch  5000 | Total: 2.777865e-04 | Data: 1.215e-04 | PDE: 9.183e-05 | Terminal: 6.167e-04 | Boundary: 2.788e-05 | r: 0.07696225 | s

In [57]:

model.eval()

with torch.no_grad():

    # PINN predicts normalized option price u = V / K
    u_validation_prediction = model(
        t_validation,
        S_validation
    )

    # Convert back to option price V
    V_validation_prediction = (
        K
        * u_validation_prediction
    )

validation_mse = torch.mean(
    (
        V_validation_prediction
        - V_validation
    ) ** 2
).item()


validation_rmse = np.sqrt(
    validation_mse
)


print("=" * 60)

print(
    "Lambda combination:"
)

print(
    f"lambda_data     = {lambda_data}"
)

print(
    f"lambda_pde      = {lambda_pde}"
)

print(
    f"lambda_terminal = {lambda_terminal}"
)

print(
    f"lambda_boundary = {lambda_boundary}"
)

print()

print(
    f"Validation MSE  = {validation_mse:.10f}"
)

print(
    f"Validation RMSE = {validation_rmse:.10f}"
)

print()

print(
    f"r_PINN     = {model.r.item():.10f}"
)

print(
    f"sigma_PINN = {model.sigma.item():.10f}"
)

print("=" * 60)

Lambda combination:
lambda_data     = 1.0
lambda_pde      = 1.0
lambda_terminal = 0.1
lambda_boundary = 0.1

Validation MSE  = 3188.0980475608
Validation RMSE = 56.4632451030

r_PINN     = 0.0667579805
sigma_PINN = 0.0801575023
